# Deep Learning Lab — Part 1: CNN, Faster R-CNN and transfer learning
**Université Abdelmalek Essaâdi • FST Tanger • Département Génie Informatique**  
Cycle: LSI • Deep Learning • Pr. ELAACHAK LOTFI  
Student: Saber Ben Hamda

Run cells in order in **Google Colab with a GPU**. No Kaggle credentials are required: torchvision downloads the original MNIST dataset corresponding to the linked Kaggle mirror. Internet is needed for data and pretrained weights. Results are computed by running the notebook; no scores are fabricated.

`QUICK=True` runs a stratified 6,000-image pilot; set it to `False` before the final report (54,000 train / 6,000 validation / 10,000 test). Every model uses identical indices. Checkpoints use validation loss; the test set is evaluated only after fitting. GPU time includes training and validation, excludes model/weight download and final testing. Colab session limits may require separate sessions; download outputs before disconnecting.

Faster R-CNN is an **object detector**, so MNIST is adapted to one digit per image with a box inferred from nonzero pixels. Report its class accuracy and macro-F1 on the same digit labels, plus localization metrics. Its composite training loss is not comparable numerically with classifiers' cross-entropy. Transfer models use ImageNet weights; their pretraining and larger inputs make this a practical comparison, not an equal-compute architecture study.


## 1. Setup, reproducible splits and shared training utilities
Regularization: AdamW weight decay 1e-4, dropout, validation-based checkpoint selection. Float32 training is used for clarity.

In [ ]:
import os, json, time, random, copy, gc, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))
# Quick mode is a balanced, stratified pilot. Use False for the final experiment.
QUICK = True
EPOCHS = 3 if QUICK else 10
N_TRAIN = 6000 if QUICK else 54000
N_VAL = 1000 if QUICK else 6000
OUT = Path('/content/mnist_lab'); OUT.mkdir(exist_ok=True)
raw_train = datasets.MNIST('/content/data', train=True, download=True)
raw_test = datasets.MNIST('/content/data', train=False, download=True)
y_all = np.array(raw_train.targets)
train_ids, val_ids = train_test_split(np.arange(60000), test_size=6000, stratify=y_all, random_state=SEED)
def select(ids, n):
    if n == len(ids): return np.array(ids)
    return train_test_split(ids, train_size=n, stratify=y_all[ids], random_state=SEED)[0]
train_ids, val_ids = select(train_ids,N_TRAIN), select(val_ids,N_VAL)
test_ids = np.arange(10000)  # identical official test set for every model
np.savez(OUT/'split.npz', train=train_ids, val=val_ids, test=test_ids)
class Digits(Dataset):
    def __init__(self, raw, ids, transform): self.raw,self.ids,self.transform=raw,ids,transform
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        x,y=self.raw[int(self.ids[i])]
        return self.transform(x), y
def loaders(transform, batch=128):
    return [DataLoader(Digits(raw,ids,transform),batch_size=batch,shuffle=(j==0),num_workers=2,pin_memory=True)
            for j,(raw,ids) in enumerate([(raw_train,train_ids),(raw_train,val_ids),(raw_test,test_ids)])]
def sync(): torch.cuda.synchronize()
criterion=nn.CrossEntropyLoss()
def evaluate(model, loader):
    model.eval(); ys=[]; ps=[]; total=0.; count=0
    with torch.inference_mode():
        for x,y in loader:
            x,y=x.to(device),y.to(device); logits=model(x)
            total+=criterion(logits,y).item()*len(y); count+=len(y)
            ys.extend(y.cpu().tolist()); ps.extend(logits.argmax(1).cpu().tolist())
    return {'accuracy':accuracy_score(ys,ps),'macro_f1':f1_score(ys,ps,labels=list(range(10)),average='macro',zero_division=0),'loss':total/count},ys,ps
def fit(model, name, ls, lr=1e-3):
    model=model.to(device)
    opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=lr,weight_decay=1e-4)
    history=[]; best=float('inf'); best_state=None
    sync(); started=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train(); total=correct=count=0
        for x,y in tqdm(ls[0],desc=f'{name} {epoch+1}/{EPOCHS}',leave=False):
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
            logits=model(x); loss=criterion(logits,y); loss.backward(); opt.step()
            total+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item(); count+=len(y)
        val,_,_=evaluate(model,ls[1])
        history.append({'epoch':epoch+1,'train_loss':total/count,'train_accuracy':correct/count,'val_loss':val['loss'],'val_accuracy':val['accuracy']})
        print(history[-1])
        if val['loss']<best:
            best=val['loss']; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    sync(); seconds=time.perf_counter()-started
    model.load_state_dict(best_state)
    metrics,ys,ps=evaluate(model,ls[2])
    metrics.update(model=name,training_seconds=seconds,loss_kind='test cross entropy')
    torch.save(best_state,OUT/f'{name}.pt')
    pd.DataFrame(history).to_csv(OUT/f'{name}_history.csv',index=False)
    fig,axs=plt.subplots(1,2,figsize=(10,3))
    h=pd.DataFrame(history); h.plot(x='epoch',y=['train_loss','val_loss'],ax=axs[0]); h.plot(x='epoch',y=['train_accuracy','val_accuracy'],ax=axs[1])
    plt.tight_layout(); plt.savefig(OUT/f'{name}_curves.png'); plt.show()
    ConfusionMatrixDisplay(confusion_matrix(ys,ps,labels=list(range(10)))).plot(); plt.title(name); plt.savefig(OUT/f'{name}_confusion.png'); plt.show()
    print(classification_report(ys,ps,digits=4,zero_division=0))
    return model,metrics
config={'seed':SEED,'quick':QUICK,'epochs':EPOCHS,'train_n':len(train_ids),'val_n':len(val_ids),'test_n':len(test_ids),'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'torchvision':torchvision.__version__}
(OUT/'config.json').write_text(json.dumps(config,indent=2))


## 2. Custom CNN
Two 3×3 convolutions (stride 1, padding 1), ReLU and 2×2 max pooling (stride 2), followed by a 128-unit fully connected layer and dropout 0.3. Input: 1×28×28; output: 10 logits.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Flatten(),
            nn.Linear(64*7*7,128),nn.ReLU(),nn.Dropout(.3),nn.Linear(128,10))
    def forward(self,x): return self.net(x)
small_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.1307,),(.3081,))])
cnn, row=fit(CNN(),'CNN',loaders(small_tf))
results=[row]
del cnn; gc.collect(); torch.cuda.empty_cache()


## 3. Fine-tune ImageNet VGG16 and AlexNet
Replicate grayscale into RGB, resize to 224×224 and apply ImageNet normalization. Keep the pretrained classifier except for its last layer. Fine-tune the last convolution block and the new output layer; freeze earlier features and hidden classifier layers to reduce memory. Learning rate: 1e-4. This updates convolution weights, so it is fine-tuning, not only feature extraction. VGG16 weights require approximately 528 MB to download.

In [ ]:
rgb_tf=transforms.Compose([transforms.Resize((224,224)),transforms.Grayscale(3),transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406),(.229,.224,.225))])
for name,builder,weights,start in [
    ('VGG16',models.vgg16,models.VGG16_Weights.DEFAULT,24),
    ('AlexNet',models.alexnet,models.AlexNet_Weights.DEFAULT,10)]:
    model=builder(weights=weights)
    for p in model.parameters(): p.requires_grad=False
    model.classifier[6]=nn.Linear(model.classifier[6].in_features,10)
    for layer in list(model.features.children())[start:]:
        for p in layer.parameters(): p.requires_grad=True
    model,row=fit(model,name,loaders(rgb_tf,batch=32),lr=1e-4)
    results.append(row)
    del model; gc.collect(); torch.cuda.empty_cache()


## 4. Faster R-CNN: single-digit detection
Use a small, randomly initialized CNN backbone with a real region proposal network, ROI Align and Faster R-CNN heads. Inputs are RGB resized internally to 64×64. Labels are shifted by +1 because 0 is background. Anchors: 8, 16, 32 pixels; aspect ratios: 0.5, 1, 2. The foreground box is the extent of pixels > 0 in the original image. This threshold is a dataset convention, not manual localization ground truth.

For image classification, take the highest-scoring detection; no detection becomes -1 and counts as an error. All ten classes are included in macro-F1. Localization uses this same detection: mean IoU (misses count as zero), and proportion with correct class and IoU ≥ 0.5. These are **not COCO mAP**.

In [ ]:
from collections import OrderedDict
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign, box_iou
class DetectionDigits(Dataset):
    def __init__(self,raw,ids): self.raw,self.ids=raw,ids
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        pil,y=self.raw[int(self.ids[i])]; x=transforms.ToTensor()(pil)
        coords=torch.nonzero(x[0]>0)
        ymin,xmin=coords.min(0).values; ymax,xmax=coords.max(0).values
        box=torch.tensor([[xmin,ymin,xmax+1,ymax+1]],dtype=torch.float32)
        return x.repeat(3,1,1),{'boxes':box,'labels':torch.tensor([y+1],dtype=torch.int64)}
def collate(batch): return tuple(zip(*batch))
dls=[DataLoader(DetectionDigits(raw,ids),batch_size=32,shuffle=(j==0),num_workers=2,collate_fn=collate)
     for j,(raw,ids) in enumerate([(raw_train,train_ids),(raw_train,val_ids),(raw_test,test_ids)])]
backbone=nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1),nn.ReLU(),
    nn.Conv2d(32,64,3,stride=2,padding=1),nn.ReLU())
backbone.out_channels=64
anchors=AnchorGenerator(sizes=((8,16,32),),aspect_ratios=((.5,1.,2.),))
detector=FasterRCNN(backbone,num_classes=11,min_size=64,max_size=64,
    image_mean=[0,0,0],image_std=[1,1,1],rpn_anchor_generator=anchors,
    box_roi_pool=MultiScaleRoIAlign(featmap_names=['0'],output_size=7,sampling_ratio=2),
    rpn_pre_nms_top_n_train=300,rpn_post_nms_top_n_train=100,
    rpn_pre_nms_top_n_test=200,rpn_post_nms_top_n_test=100,
    box_score_thresh=0.0,box_detections_per_img=10).to(device)
def detection_eval(model,loader):
    model.eval(); ys=[]; ps=[]; ious=[]; success=[]
    with torch.inference_mode():
        for xs,ts in loader:
            outputs=model([x.to(device) for x in xs])
            for out,t in zip(outputs,ts):
                true=int(t['labels'][0])-1; ys.append(true)
                if len(out['scores']):
                    k=out['scores'].argmax(); pred=int(out['labels'][k])-1
                    iou=float(box_iou(out['boxes'][k:k+1].cpu(),t['boxes'])[0,0])
                else: pred=-1; iou=0.
                ps.append(pred); ious.append(iou); success.append(pred==true and iou>=.5)
    return {'accuracy':accuracy_score(ys,ps),'macro_f1':f1_score(ys,ps,labels=list(range(10)),average='macro',zero_division=0),
            'mean_iou':float(np.mean(ious)),'class_and_iou50':float(np.mean(success))},ys,ps
opt=torch.optim.AdamW(detector.parameters(),lr=1e-3,weight_decay=1e-4)
best=-1.; history=[]; sync(); start=time.perf_counter()
for epoch in range(EPOCHS):
    detector.train(); loss_total=0.; count=0; components={}
    for xs,ts in tqdm(dls[0],desc=f'Faster R-CNN {epoch+1}',leave=False):
        xs=[x.to(device) for x in xs]; ts=[{k:v.to(device) for k,v in t.items()} for t in ts]
        opt.zero_grad(set_to_none=True); losses=detector(xs,ts); loss=sum(losses.values())
        assert torch.isfinite(loss), losses
        loss.backward(); opt.step(); loss_total+=loss.item()*len(xs); count+=len(xs)
        for k,v in losses.items(): components[k]=components.get(k,0.)+v.item()*len(xs)
    val,_,_=detection_eval(detector,dls[1])
    row={'epoch':epoch+1,'train_detection_loss':loss_total/count,'val_accuracy':val['accuracy'],**{k:v/count for k,v in components.items()}}
    history.append(row); print(row)
    if val['accuracy']>best:
        best=val['accuracy']; state={k:v.detach().cpu().clone() for k,v in detector.state_dict().items()}
sync(); elapsed=time.perf_counter()-start
detector.load_state_dict(state)
row,ys,ps=detection_eval(detector,dls[2])
# Eval-mode Faster R-CNN returns predictions, not a cross-entropy test loss.
row.update(model='FasterRCNN',loss=float('nan'),loss_kind='N/A: detector objective',training_seconds=elapsed)
results.append(row)
torch.save(state,OUT/'FasterRCNN.pt')
h=pd.DataFrame(history); h.to_csv(OUT/'FasterRCNN_history.csv',index=False)
h.plot(x='epoch',y=['train_detection_loss','loss_classifier','loss_box_reg','loss_objectness','loss_rpn_box_reg']); plt.savefig(OUT/'FasterRCNN_loss.png'); plt.show()
h.plot(x='epoch',y='val_accuracy'); plt.show()
print(row)
print(classification_report(ys,ps,labels=list(range(10)),digits=4,zero_division=0))
# Include -1 explicitly to visualize missed detections.
ConfusionMatrixDisplay(confusion_matrix(ys,ps,labels=[-1]+list(range(10))),display_labels=['miss']+list(range(10))).plot()
plt.savefig(OUT/'FasterRCNN_confusion.png'); plt.show()
del detector; gc.collect(); torch.cuda.empty_cache()


## 5. Compare and export
Compare accuracy, macro-F1, time, and classifier cross-entropy. Use the separate detector loss curve to discuss convergence; do not rank its loss against cross-entropy. Check overfitting by comparing training and validation curves. Training time includes validation passes for both approaches, but their validation calculations differ.

In [ ]:
table=pd.DataFrame(results)
table.to_csv(OUT/'part1_results.csv',index=False)
display(table)
table.set_index('model')[['accuracy','macro_f1']].plot.bar(ylim=(0,1),rot=0); plt.tight_layout(); plt.savefig(OUT/'comparison.png'); plt.show()
print('Highest test accuracy:',table.loc[table.accuracy.idxmax(),'model'])
print('Fastest measured training:',table.loc[table.training_seconds.idxmin(),'model'])
print('Discuss whether any small accuracy gain justifies the additional compute and pretrained data.')


## 6. Brief synthesis and README report
Complete the personal reflection after examining your actual results. The report below is generated from measured outputs; add your own interpretation and GitHub screenshots. Upload this notebook, README and result files to your assignment repository. Do not commit raw datasets or large checkpoints.

In [ ]:
report='# MNIST computer vision lab — Part 1\n\n'
report+='Student: Saber Ben Hamda | FST Tanger | Deep Learning\n\n'
report+='Configuration: '+json.dumps(config)+'\n\n'
report+='```text\n'+table.to_string(index=False)+'\n```\n\n'
report+='## Method\nIdentical stratified splits; validation checkpoint selection; official 10,000-image test set. CNN and Faster R-CNN trained from scratch; VGG16/AlexNet initialized on ImageNet and fine-tuned. Detector boxes inferred from nonzero pixels.\n\n'
report+='## Interpretation to complete\n- Explain the observed accuracy/F1 and training time ranking.\n- Discuss train/validation gaps and misclassified digits.\n- Explain why detection loss and classification loss cannot be ranked together.\n- State limitations: pilot size if enabled, unequal input resolution, ImageNet pretraining, single seed and synthetic detection boxes.\n\n'
report+='## My synthesis (complete in your own words)\nI learned how to ...\nThe most important result I observed was ...\nA limitation I would address next is ...\n'
(OUT/'README_part1.md').write_text(report)
import shutil
shutil.make_archive('/content/part1_outputs','zip',OUT)
from google.colab import files
files.download('/content/part1_outputs.zip')


## References
- [MNIST Kaggle mirror](https://www.kaggle.com/datasets/hojjatk/mnist-dataset)
- [PyTorch Faster R-CNN](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html)
- [VGG16](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html) and [AlexNet](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html)